In [13]:
import yfinance as yf
import pandas as pd
import ta  # pip install ta

import datetime



startdt="2025-01-01"
enddt = datetime.date.today().strftime("%Y-%m-%d")
# Fetch stock data
enddt = datetime.date.today().strftime("%Y-%m-%d")
data = yf.download("TCS.NS", start=startdt, end=enddt)

# Ensure Close is a Series (1D)
close = data["Close"].squeeze()

# RSI
data["RSI"] = ta.momentum.RSIIndicator(close, window=14).rsi()

# MACD
macd = ta.trend.MACD(close)
data["MACD"] = macd.macd()
data["MACD_Signal"] = macd.macd_signal()

# Bollinger Bands
bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
data["BB_High"] = bb.bollinger_hband()
data["BB_Low"] = bb.bollinger_lband()

print(data.tail())


/tmp/ipykernel_18997/3881478293.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("TCS.NS", start=startdt, end=enddt)
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open   Volume  \
Ticker           TCS.NS       TCS.NS       TCS.NS       TCS.NS   TCS.NS   
Date                                                                      
2025-12-08  3236.500000  3259.899902  3208.600098  3243.000000  2544725   
2025-12-09  3208.300049  3233.699951  3181.199951  3230.000000  2734893   
2025-12-10  3189.199951  3225.000000  3178.100098  3225.000000  1386849   
2025-12-11  3191.899902  3208.399902  3153.100098  3205.000000  1914362   
2025-12-12  3220.500000  3223.000000  3185.699951  3197.699951  1146554   

Price             RSI       MACD MACD_Signal      BB_High       BB_Low  
Ticker                                                                  
Date                                                                    
2025-12-08  69.592216  43.414055   34.558802  3238.272056  3049.057924  
2025-12-09  63.144947  42.942988   36.235639  3239.234473  3064.225513  
2025-12-10  59.148011  40.560885  

In [14]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# OR suppress only FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning)

# OR suppress only UserWarning
warnings.filterwarnings("ignore", category=UserWarning)

In [15]:
import yfinance as yf
import pandas as pd
import ta
import datetime

# -----------------------------
# Define Stocks & Date Range
# -----------------------------
stocks = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ACC.NS", "BAJFINANCE.NS","CUMMINSIND.NS"]
start_date = "2025-01-01"
end_date =  datetime.date.today().strftime("%Y-%m-%d")
# Store results
screen_results = {}

for ticker in stocks:
    print(f"\n📊 Processing {ticker}...")
    
    # Fetch stock data
    data = yf.download(ticker, start=start_date, end=end_date)
    close = data["Close"].squeeze()  # Ensure it's 1D
    
    # -----------------------------
    # Indicators
    # -----------------------------
    # RSI
    data["RSI"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    # MACD
    macd = ta.trend.MACD(close)
    data["MACD"] = macd.macd()
    data["MACD_Signal"] = macd.macd_signal()

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    data["BB_High"] = bb.bollinger_hband()
    data["BB_Low"] = bb.bollinger_lband()

    # -----------------------------
    # Screening Logic
    # -----------------------------
    latest = data.iloc[-1]  # last available row

    # Convert to scalars
    rsi_val = float(latest["RSI"])
    close_val = float(latest["Close"])
    macd_val = float(latest["MACD"])
    macd_sig = float(latest["MACD_Signal"])
    bb_low = float(latest["BB_Low"])
    bb_high = float(latest["BB_High"])

    # Simple rules
    signal = "HOLD"
    if rsi_val < 30 and close_val < bb_low:
        signal = "BUY"
    elif rsi_val > 70 and close_val > bb_high:
        signal = "SELL"

    # Save summary
    screen_results[ticker] = {
        "Last Close": round(close_val, 2),
        "RSI": round(rsi_val, 2),
        "MACD": round(macd_val, 2),
        "MACD_Signal": round(macd_sig, 2),
        "BB_Low": round(bb_low, 2),
        "BB_High": round(bb_high, 2),
        "Recommendation": signal
    }

# -----------------------------
# Final Screener Results
# -----------------------------
results_df = pd.DataFrame(screen_results).T
print("\n📌 Stock Screening Results:")
print(results_df)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


📊 Processing RELIANCE.NS...

📊 Processing TCS.NS...

📊 Processing HDFCBANK.NS...

📊 Processing ACC.NS...

📊 Processing BAJFINANCE.NS...

📊 Processing CUMMINSIND.NS...

📌 Stock Screening Results:
              Last Close    RSI   MACD MACD_Signal   BB_Low  BB_High  \
RELIANCE.NS       1556.5   62.8  15.24       19.36  1512.92  1573.77   
TCS.NS            3220.5  63.51  38.64       37.62  3077.35  3251.92   
HDFCBANK.NS       1001.5  53.21   2.03        2.66   987.52   1011.0   
ACC.NS            1771.7  32.85  -19.4      -10.76  1758.37  1907.81   
BAJFINANCE.NS     1017.3  48.19  -3.38       -2.85    989.2  1047.11   
CUMMINSIND.NS     4600.2  68.35  75.89        72.5  4236.94  4629.71   

              Recommendation  
RELIANCE.NS             HOLD  
TCS.NS                  HOLD  
HDFCBANK.NS             HOLD  
ACC.NS                  HOLD  
BAJFINANCE.NS           HOLD  
CUMMINSIND.NS           HOLD  


In [16]:
import yfinance as yf
import pandas as pd
import ta

# List of stocks to screen
stocks = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "CUMMINSIND.NS"]

# Prepare results
results = []

for symbol in stocks:
    print(f"\n📊 Processing {symbol}...")

    # Download 2 years of daily data with auto_adjust=True (adjusted close)
    df = yf.download(symbol, start="2024-01-01", end="2025-08-19", auto_adjust=True)

    if df.shape[0] < 30:
        print("Not enough data for reliable indicators.")
        continue

    # Use Adjusted Close for indicator calculations
    close = df["Close"].squeeze()

    # RSI
    df["RSI"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    # MACD
    macd = ta.trend.MACD(close)
    df["MACD"] = macd.macd()
    df["MACD_signal"] = macd.macd_signal()

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(close, window=20)
    df["BB_High"] = bb.bollinger_hband()
    df["BB_Low"] = bb.bollinger_lband()

    # Drop NaNs to ensure valid latest row
    df = df.dropna()

    # Get latest row
    last = df.iloc[-1]
    close_val = last["Close"].item()
    rsi_val = last["RSI"].item()
    macd_val = last["MACD"].item()
    macd_signal = last["MACD_signal"].item()
    bb_high = last["BB_High"].item()
    bb_low = last["BB_Low"].item()

    # Simple scoring system
    score = 0
    # RSI rules
    if rsi_val < 30: score += 1
    elif rsi_val > 70: score -= 1
    # MACD rules
    if macd_val > macd_signal: score += 1
    else: score -= 1
    # Bollinger Bands rules
    if close_val < bb_low: score += 1
    elif close_val > bb_high: score -= 1

    # Recommendation
    if score >= 2:
        decision = "Strong Buy"
    elif score == 1:
        decision = "Buy"
    elif score == 0:
        decision = "Hold"
    elif score == -1:
        decision = "Sell"
    else:
        decision = "Strong Sell"

    # Append to results
    results.append({
        "Stock": symbol,
        "Close": close_val,
        "RSI": rsi_val,
        "MACD": macd_val,
        "MACD_Signal": macd_signal,
        "BB_Low": bb_low,
        "BB_High": bb_high,
        "Score": score,
        "Recommendation": decision
    })

# Display results as DataFrame
results_df = pd.DataFrame(results)
print("\n📌 Stock Screening Results:")
print(results_df)



📊 Processing RELIANCE.NS...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


📊 Processing TCS.NS...

📊 Processing HDFCBANK.NS...

📊 Processing CUMMINSIND.NS...

📌 Stock Screening Results:
           Stock        Close        RSI       MACD  MACD_Signal       BB_Low  \
0    RELIANCE.NS  1381.699951  41.447158 -19.940808   -20.364417  1359.477011   
1         TCS.NS  3000.010986  28.683943 -66.471856   -73.265589  2953.315237   
2    HDFCBANK.NS  1001.799988  54.262381   1.220976     2.508161   983.638085   
3  CUMMINSIND.NS  3783.399902  71.531714  93.616709    84.493240  3444.899207   

       BB_High  Score Recommendation  
0  1422.233597      1            Buy  
1  3164.796213      2     Strong Buy  
2  1015.448249     -1           Sell  
3  3842.930724      0           Hold  


In [17]:
import yfinance as yf
import pandas as pd
import ta
from datetime import datetime, timedelta

# Stocks to analyze
stocks = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ACC.NS", "BAJFINANCE.NS", "CUMMINSIND.NS"]

# Last 1 year
end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d')

results = []

for symbol in stocks:
    print(f"\n📊 Processing {symbol}...")
    df = yf.download(symbol, start=start_date, end=end_date, auto_adjust=True)

    if df.shape[0] < 30:
        print(f"Not enough data for {symbol}. Skipping...")
        continue

    close = df["Close"].squeeze()

    # Calculate indicators
    df["RSI"] = ta.momentum.RSIIndicator(close, window=14).rsi()
    macd = ta.trend.MACD(close)
    df["MACD"] = macd.macd()
    df["MACD_Signal"] = macd.macd_signal()
    bb = ta.volatility.BollingerBands(close, window=20)
    df["BB_High"] = bb.bollinger_hband()
    df["BB_Low"] = bb.bollinger_lband()

    df = df.dropna()
    latest = df.iloc[-1]

    # --- Recommendation logic ---
    score = 0
    rsi_val = latest["RSI"].item()
    macd_val = latest["MACD"].item()
    macd_signal = latest["MACD_Signal"].item()
    close_val = latest["Close"].item()
    bb_low = latest["BB_Low"].item()
    bb_high = latest["BB_High"].item()


    # RSI rules
    if rsi_val < 30:
        score += 1
    elif rsi_val > 70:
        score -= 1

    # MACD rules
    if macd_val > macd_signal:
        score += 1
    else:
        score -= 1

    # Bollinger Bands rules
    if close_val < bb_low:
        score += 1
    elif close_val > bb_high:
        score -= 1

    # Final Recommendation
    if score > 0:
        recommendation = "BUY"
    elif score < 0:
        recommendation = "SELL"
    else:
        recommendation = "HOLD"

    results.append({
        "Stock": symbol,
        "Close": close_val,
        "RSI": rsi_val,
        "MACD": macd_val,
        "MACD_Signal": macd_signal,
        "BB_High": bb_high,
        "BB_Low": bb_low,
        "Score": score,
        "Recommendation": recommendation
    })

# Display results
results_df = pd.DataFrame(results)
print("\n📌 Stock Screening Results (Last 1 Year):")
print(results_df)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



📊 Processing RELIANCE.NS...

📊 Processing TCS.NS...

📊 Processing HDFCBANK.NS...


[*********************100%***********************]  1 of 1 completed



📊 Processing ACC.NS...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



📊 Processing BAJFINANCE.NS...

📊 Processing CUMMINSIND.NS...

📌 Stock Screening Results (Last 1 Year):
           Stock        Close        RSI       MACD  MACD_Signal      BB_High  \
0    RELIANCE.NS  1556.500000  62.800981  15.235234    19.361040  1573.768829   
1         TCS.NS  3220.500000  63.510649  38.635432    37.623159  3251.920869   
2    HDFCBANK.NS  1001.500000  53.206263   2.025338     2.655944  1010.996114   
3         ACC.NS  1771.699951  32.848359 -19.401036   -10.761760  1907.809335   
4  BAJFINANCE.NS  1017.299988  48.186361  -3.384709    -2.851717  1047.114599   
5  CUMMINSIND.NS  4600.200195  68.346147  75.886131    72.504840  4629.706307   

        BB_Low  Score Recommendation  
0  1512.921185     -1           SELL  
1  3077.349102      1            BUY  
2   987.518871     -1           SELL  
3  1758.370633     -1           SELL  
4   989.195404     -1           SELL  
5  4236.943693      1            BUY  


In [18]:
import yfinance as yf
import pandas as pd
import ta
from datetime import datetime, timedelta

# Stocks list
stocks = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ACC.NS", "BAJFINANCE.NS", "CUMMINSIND.NS"]

# Dates
end_date = datetime.today()
start_date = end_date - timedelta(days=365)

results = []

for ticker in stocks:
    print(f"📊 Processing {ticker}...")

    # Download data
    df = yf.download(ticker, start=start_date, end=end_date)

    if df.empty:
        print(f"No data for {ticker}")
        continue

    # Ensure Close is 1D Series
    close = df["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]  # pick the first column
    close = close.astype(float)

    # Calculate indicators
    df["RSI"] = ta.momentum.RSIIndicator(close, window=14).rsi()
    macd = ta.trend.MACD(close)
    df["MACD"] = macd.macd()
    df["MACD_Signal"] = macd.macd_signal()
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    df["BB_High"] = bb.bollinger_hband()
    df["BB_Low"] = bb.bollinger_lband()

    # Most recent row
    latest = df.iloc[-1]

    rsi_val = float(latest["RSI"])
    macd_val = float(latest["MACD"])
    macd_signal = float(latest["MACD_Signal"])
    close_val = float(latest["Close"])
    bb_low = float(latest["BB_Low"])
    bb_high = float(latest["BB_High"])

    # Recommendation: all 3 align for BUY/SELL
    recommendation = "HOLD"
    if (rsi_val < 30) and (macd_val > macd_signal) and (close_val < bb_low):
        recommendation = "BUY"
    elif (rsi_val > 70) and (macd_val < macd_signal) and (close_val > bb_high):
        recommendation = "SELL"

    results.append({
        "Stock": ticker,
        "Last Close": close_val,
        "RSI": rsi_val,
        "MACD": macd_val,
        "MACD_Signal": macd_signal,
        "BB_Low": bb_low,
        "BB_High": bb_high,
        "Recommendation": recommendation
    })

# Display results
screening_df = pd.DataFrame(results)
print("\n📌 Stock Screening Results:")
print(screening_df)


📊 Processing RELIANCE.NS...


[*********************100%***********************]  1 of 1 completed


📊 Processing TCS.NS...


[*********************100%***********************]  1 of 1 completed


📊 Processing HDFCBANK.NS...


[*********************100%***********************]  1 of 1 completed


... (3 outputs left out)

[*********************100%***********************]  1 of 1 completed


📊 Processing CUMMINSIND.NS...


[*********************100%***********************]  1 of 1 completed


📌 Stock Screening Results:
           Stock   Last Close        RSI       MACD  MACD_Signal       BB_Low  \
0    RELIANCE.NS  1556.500000  62.800981  15.235234    19.361040  1512.921185   
1         TCS.NS  3220.500000  63.510649  38.635432    37.623159  3077.349102   
2    HDFCBANK.NS  1001.500000  53.206263   2.025338     2.655944   987.518871   
3         ACC.NS  1771.699951  32.848359 -19.401036   -10.761760  1758.370633   
4  BAJFINANCE.NS  1017.299988  48.186361  -3.384709    -2.851717   989.195404   
5  CUMMINSIND.NS  4600.200195  68.346147  75.886131    72.504840  4236.943693   

       BB_High Recommendation  
0  1573.768829           HOLD  
1  3251.920869           HOLD  
2  1010.996114           HOLD  
3  1907.809335           HOLD  
4  1047.114599           HOLD  
5  4629.706307           HOLD  


In [19]:
import yfinance as yf
import pandas as pd
import ta
from datetime import datetime, timedelta

# Example tickers
tickers = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "HINDUNILVR.NS",
    "ICICIBANK.NS", "KOTAKBANK.NS", "SBIN.NS", "BAJFINANCE.NS", "BHARTIARTL.NS","CUMMINSIND.NS"
]

end_date = datetime.today()
start_date = end_date - timedelta(days=365)

results = []

for ticker in tickers:
    try:
        df = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
        if df.empty:
            print(f"No data for {ticker}")
            continue

        # Ensure Close is 1D Series
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        close = close.astype(float)

        # RSI
        rsi = ta.momentum.RSIIndicator(close, window=14).rsi()
        # MACD
        macd = ta.trend.MACD(close)
        macd_line = macd.macd()
        macd_signal = macd.macd_signal()
        # Bollinger Bands
        bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
        bb_high = bb.bollinger_hband()
        bb_low = bb.bollinger_lband()

        latest = {
            'Close': close.iloc[-1],
            'RSI': rsi.iloc[-1],
            'MACD': macd_line.iloc[-1],
            'MACD_Signal': macd_signal.iloc[-1],
            'BB_High': bb_high.iloc[-1],
            'BB_Low': bb_low.iloc[-1]
        }

        # Momentum scoring
        score = 0
        score += 1 if latest['RSI'] < 30 else (-1 if latest['RSI'] > 70 else 0)
        score += 1 if latest['MACD'] > latest['MACD_Signal'] else -1
        ma50 = close.rolling(50).mean().iloc[-1]
        ma200 = close.rolling(200).mean().iloc[-1]
        score += 1 if latest['Close'] > ma50 and latest['Close'] > ma200 else -1
        score += 1 if latest['Close'] <= latest['BB_Low']*1.03 else (-1 if latest['Close'] >= latest['BB_High']*0.97 else 0)

        # Recommendation
        if score >= 4:
            recommendation = "STRONG BUY"
        elif score == 3:
            recommendation = "BUY"
        elif score == 2:
            recommendation = "HOLD"
        else:
            recommendation = "SELL"

        latest.update({'Score': score, 'Recommendation': recommendation, 'Stock': ticker})
        results.append(latest)

    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        continue

momentum_df = pd.DataFrame(results)
pd.set_option('display.max_columns', None)
print(momentum_df)


          Close        RSI       MACD  MACD_Signal      BB_High       BB_Low  \
0   1556.500000  62.800981  15.235234    19.361040  1573.768829  1512.921185   
1   3220.500000  63.510649  38.635432    37.623158  3251.920869  3077.349102   
2   1001.500000  53.206263   2.025338     2.655944  1010.996114   987.518871   
3   1598.199951  62.322935  25.698084    24.958603  1631.511918  1497.278073   
4   2260.600098  25.099459 -42.268466   -26.126419  2531.715662  2267.684363   
5   1366.000000  46.306800  -0.124862     2.145465  1399.965982  1355.874008   
6   2176.600098  63.417946  14.389200     8.620956  2181.139184  2064.460791   
7    963.150024  53.430406   5.288584     8.959171   989.924736   948.635250   
8   1017.299988  48.186361  -3.384709    -2.851717  1047.114599   989.195404   
9   2083.399902  49.213192   1.739870    11.561328  2179.188547  2047.981473   
10  4600.200195  68.346147  75.886131    72.504840  4629.706307  4236.943693   

    Score Recommendation          Stock